# cross-validation

### why train/test split is not enpugh.
Train/test split-> where we have been splitting out data, 80% training, 20% testing, check accuracy.

- Problem 1: Luck, that 20% test set might habe been usually or unusually hard.
- Problem 2: Small data, e,e if you only have 100 rows, you test is just 20 rows, that's too small to trust
- Problem 3: Different results every time e.g if i change the random split, I may get a completly different 
accuracy.

-> One test = unrealible.
-> Multiple tests = reliable.


## K-fold cross validation
1. Split the full dataset into K equal-sized 'folds' (K=5 or k=10 are most common).
2. Train on K-1 folds, evaluate on the remaining fold - record the score.
3. Repeate K times, each time using a different fold as the validation set.
4. The final score is the mean of the K scores. The standard deviation tells us how consistent
model is.

### The objective is we want A:
- High average.
- Low variation i.e low standard deviation.

## Stratified K-fold - Important for classification.
In regular K-fold, each fold is a random 1/k slice of the data.
If your dataset is 90% class 0 and 10% class 1, some folds might have no class 1 examples at all.

Stratified K-fold ensures each fold has the same class proportions as the full dataset.

```Always use Stratified for classification```

In [1]:
# Loading the data.
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target


In [2]:
X.shape

(569, 30)

In [11]:
# Basic K-fold (not ideal for classification)
from sklearn.model_selection import (StratifiedKFold, KFold, cross_val_score)
from sklearn.ensemble import RandomForestClassifier

kf = KFold(n_splits=5, random_state=6, shuffle=True)
scores = cross_val_score(RandomForestClassifier(random_state=42),X, y, cv=kf,)

# find the mean of the accuracy scores
print(f'Mean accuracy score: {scores.mean():.3f} | Standard Deviation:{scores.std():.3f}')
scores

Mean accuracy score: 0.965 | Standard Deviation:0.016


array([0.96491228, 0.98245614, 0.94736842, 0.94736842, 0.98230088])

## Interpretation: 

### Mean score
- 0.90 - 1.00 Excellent
- 0.70 - 0.90 Good/stable
- Below 0.50 Poor.

# Standard deviation score
- Below 0.02 (Best)
- 0.02 - 0.05 (Acceptable)
- Above 0.10 (High/risky)
# Coefficient of variation score:
CV = SD/ mean.
If your cv is below 5%, your model is best.
If your cv is above 15%, dont rely on that model, do some modifications.

# Example
Model A -> 0.80 +- 0.02
Model B -> 0.85 +- 0.12, your model will be unstable with accuracy 

In [14]:
# Stratified K-fold (Better for imbalanced datasets)
from tracemalloc import start

from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=6)
scores = cross_val_score(RandomForestClassifier(random_state=42),X,y, cv=skf)

print(f'Mean score: {scores.mean():.3f} | std: {scores.std():.3f}')
scores

Mean score: 0.963 | std: 0.017


array([0.98245614, 0.95614035, 0.93859649, 0.98245614, 0.95575221])

In [15]:
# Cross Validate -> Multiple 
from sklearn.model_selection import cross_validate

results = cross_validate(
    RandomForestClassifier(random_state=42),
    X,
    y,
    cv=skf,
    scoring=['accuracy','precision','recall','f1'],
    return_train_score=True
)
results

{'fit_time': array([0.40779972, 0.60928464, 0.46838355, 0.53057194, 0.42825723]),
 'score_time': array([0.04258084, 0.04075217, 0.03793716, 0.04059315, 0.02748466]),
 'test_accuracy': array([0.98245614, 0.95614035, 0.93859649, 0.98245614, 0.95575221]),
 'train_accuracy': array([1., 1., 1., 1., 1.]),
 'test_precision': array([0.97260274, 0.95833333, 0.95774648, 0.97297297, 0.95833333]),
 'train_precision': array([1., 1., 1., 1., 1.]),
 'test_recall': array([1.        , 0.97183099, 0.94444444, 1.        , 0.97183099]),
 'train_recall': array([1., 1., 1., 1., 1.]),
 'test_f1': array([0.98611111, 0.96503497, 0.95104895, 0.98630137, 0.96503497]),
 'train_f1': array([1., 1., 1., 1., 1.])}

In [25]:
print('Per Fold results:')
for metric in ['accuracy','precision','recall','f1']:
    test_vals = results[f'test_{metric}']
    train_vals = results[f'train_{metric}']
    print(f"{metric:12s} test={test_vals.mean():.3f}, standard deviation={test_vals.std():.3f}"
          f"train = {train_vals.mean():.3f}")

Per Fold results:
accuracy     test=0.963, standard deviation=0.017train = 1.000
precision    test=0.964, standard deviation=0.007train = 1.000
recall       test=0.978, standard deviation=0.021train = 1.000
f1           test=0.971, standard deviation=0.014train = 1.000


In [44]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

models = {
    'Logistic Regression' : Pipeline([
       ('scaler', StandardScaler()),
       ('clf', LogisticRegression())
    ]),
    'Decision Tree':DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':RandomForestClassifier(random_state=42),
    'KNN': Pipeline([
       ('scaler', StandardScaler()),
        ('clf',KNeighborsClassifier())
    ])
}

for name, model in models.items():
    sc = cross_val_score(model, X, y, cv=skf)
    print(f'{name:20s} mean= {sc.mean():.3f}, standard deviation= {sc.std():.3f}')

Logistic Regression  mean= 0.979, standard deviation= 0.007
Decision Tree        mean= 0.924, standard deviation= 0.029
Random Forest        mean= 0.963, standard deviation= 0.017
KNN                  mean= 0.967, standard deviation= 0.013
